# Run B, seed 44: ViT-B/16 with the v2 preprocessing

This notebook trains the ViT **once, with seed 44**, on the same participant split as every other
run, then tests it on the same 8,938 test clocks, compares it with the summer ViT, and writes the
record of the run. One notebook per seed, so every seed has its own file and its own record.

**Status:** **Not done yet.**

**How long:** about 35 min to copy the clocks from Drive, about 4 min to check them, then about
5 to 6 hours of training (about 22 min per epoch on an A100, it stops by itself, 25 epochs at most).

**If Colab disconnects or the laptop sleeps:** open this same notebook again and do Runtime > Run all,
without changing anything. After every epoch the whole training state is saved on Drive, so it
continues from the last finished epoch. You lose at most the epoch that was running.

**Rules**
* Runtime > Change runtime type > A100 GPU (or L4)
* Never "Save a copy in GitHub" from Colab, the repo is public. Everything is saved on Drive.
* Don't change the seed here. Seed 44 belongs to this notebook only.

**What it saves on Drive**, in `cdt-data/results_v2/vit_b16_v2_s44/`:

| file | what's in it |
|---|---|
| `RUN_RECORD.md` | the readable record: sessions, settings, test results, every epoch (written from the files below, never by hand) |
| `metrics.json` | every test number with its 95% interval |
| `history.json` | every epoch (loss, accuracy, validation kappa), written after each epoch |
| `train_log.txt` | everything the run printed, from every session |
| `sessions.json` | each Colab session: start, end, code version, GPU, command |
| `predictions_test.csv` | one row per test clock (has participant IDs, stays on Drive) |

and the model in `cdt-data/models/vit_b16_v2_s44_best.pt`.

In [1]:
# we mount Drive (the NHATS data, the models and the results are there) and get the latest code
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/cdt-alzheimer-screening'
if not os.path.exists(REPO):
    !git clone -q https://github.com/wiambenadder/cdt-alzheimer-screening.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt
!git log --oneline -1

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > A100 GPU, then Run all again'
print('GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
/content/cdt-alzheimer-screening
b7f5a43 (HEAD -> main, origin/main, origin/HEAD) Created using Colab
GPU: NVIDIA A100-SXM4-40GB


In [2]:
SEED = 44                      # this notebook is only for this seed
RUN = f'vit_b16_v2_s{SEED}'
DRIVE_DATA = '/content/drive/MyDrive/cdt-data'
WORKERS = os.cpu_count()          # reading the TIFFs is the slow part, so we use every CPU

os.environ['CDT_DATA_DIR'] = DRIVE_DATA
os.environ['CDT_MODELS_DIR'] = f'{DRIVE_DATA}/models'
os.environ['CDT_RESULTS_DIR'] = f'{DRIVE_DATA}/results_v2'
RUN_DIR = f"{os.environ['CDT_RESULTS_DIR']}/{RUN}"

import json
DONE = os.path.exists(f'{RUN_DIR}/metrics.json')
epochs_done = len(json.load(open(f'{RUN_DIR}/history.json'))) if os.path.exists(f'{RUN_DIR}/history.json') else 0
can_continue = os.path.exists(f"{os.environ['CDT_MODELS_DIR']}/{RUN}_last.pt")
if DONE:
    print(f'{RUN} is already finished, this notebook will only write its record (no data copy, no training)')
elif can_continue:
    print(f'{RUN} was cut after {epochs_done} epoch(s), it will continue from there')
else:
    print(f'{RUN} starts from epoch 0')

vit_b16_v2_s44 was cut after 24 epoch(s), it will continue from there


In [3]:
# we copy the TIFFs to the local disk first (reading 59k files from Drive during training is too
# slow), then we check every file: one that is missing or a different size than on Drive is copied
# again, then opened fully. A half-copied file crashed a training run once (Nov 3).
if DONE:
    print('already finished, no need to copy the clocks')
else:
    import shutil, time
    import pandas as pd
    from pathlib import Path
    from concurrent.futures import ThreadPoolExecutor
    from PIL import Image

    LOCAL = Path('/content/nhats_local')
    LOCAL.mkdir(exist_ok=True)
    RAW = Path(f'{DRIVE_DATA}/nhats_raw')
    t0 = time.time()
    for rf in sorted(RAW.iterdir()):
        if not rf.is_dir():
            continue
        dst = LOCAL / rf.name
        if dst.exists() and any(dst.glob('*.tif')):
            continue
        print(f'[copy] {rf.name}')
        shutil.copytree(rf, dst, dirs_exist_ok=True)
    print(f'staged in {(time.time() - t0) / 60:.1f} min')

    labels = pd.read_csv(f'{DRIVE_DATA}/labels.csv')
    labels['image_path'] = labels['image_path'].str.replace(f'{DRIVE_DATA}/nhats_raw/', str(LOCAL) + '/', regex=False)

    def check(path):
        local = Path(path)
        drive_file = RAW / local.relative_to(LOCAL)
        try:
            if not local.exists() or local.stat().st_size != drive_file.stat().st_size:
                shutil.copy2(drive_file, local)
            with Image.open(local) as im:
                im.load()
            return None
        except Exception:
            return path

    t0 = time.time()
    with ThreadPoolExecutor(16) as pool:
        bad = [p for p in pool.map(check, labels['image_path']) if p]
    print(f'checked {len(labels):,} files in {(time.time() - t0) / 60:.1f} min')
    # still broken = broken on Drive too, so we leave them out (the split is by participant,
    # so nobody moves between train, val and test). Only the count is printed, the names are IDs
    if bad:
        Path(f'{DRIVE_DATA}/broken_files.txt').write_text('\n'.join(bad))
        labels = labels[~labels['image_path'].isin(bad)]
        print(f'{len(bad)} files are broken even on Drive and are left out (list in cdt-data/broken_files.txt)')
    else:
        print('all files open fine')
    LABELS = '/content/labels_local.csv'
    labels.to_csv(LABELS, index=False)
    print(f'{len(labels):,} labels -> {LABELS}')

[copy] round_01
[copy] round_02
[copy] round_03
[copy] round_04
[copy] round_05
[copy] round_06
[copy] round_07
[copy] round_08
[copy] round_09
[copy] round_10
[copy] round_11
[copy] round_12
[copy] round_13
[copy] round_14
staged in 41.9 min
checked 59,417 files in 3.7 min
all files open fine
59,417 labels -> /content/labels_local.csv


## Training

`--resume` continues a run that was cut, and a run that is already finished is skipped, so this
cell is always safe to run again. The lines to look for: `device=cuda`, then one `[train] ep=..`
line per epoch, then `test kappa (argmax)` and `test kappa (tuned)`.

In [4]:
if not DONE:
    !python -m src.run --models vit_b16 --seeds {SEED} --select-metric val_kappa \
        --run-tag _v2 --labels {LABELS} --num-workers {WORKERS} --resume

I0000 00:00:1793815328.559215   68480 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793815328.631052   68480 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793815331.326640   68480 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793815331.327378   68480 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

## Comparison with the summer ViT and the record

The paired comparison uses the same test clocks for both models. If its interval doesn't cross 0,
the difference is real. Then the record is written from the saved files and shown below.

In [5]:
if os.path.exists(f'{RUN_DIR}/predictions_test.csv'):
    !python -m src.run --compare {RUN} vit_b16_v1_s42
!python -m src.record {RUN}

from IPython.display import Markdown, display
record = f'{RUN_DIR}/RUN_RECORD.md'
display(Markdown(open(record).read()) if os.path.exists(record) else 'no record yet')

I0000 00:00:1793817258.750679   77159 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793817258.824528   77159 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793817261.447065   77159 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793817261.447612   77159 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

# Run record: vit_b16_v2_s44

Written automatically on 2026-11-04 18:34:32 UTC by `python -m src.record`, from the files in this folder. Nothing here is typed by hand.

## Status

Finished: training (if any) and evaluation are done, see `metrics.json`.

## Sessions

| started | ended | status | code version | GPU |
|---|---|---|---|---|
| 2026-11-04 07:17:44 UTC | never written | cut (Colab disconnected or the computer turned off) | b7f5a43 Created using Colab | NVIDIA A100-SXM4-40GB |
| 2026-11-04 18:02:14 UTC | 2026-11-04 18:34:07 UTC | finished | b7f5a43 Created using Colab | NVIDIA A100-SXM4-40GB |

Command of the last session:

```
python -m src.run --models vit_b16 --seeds 44 --select-metric val_kappa --run-tag _v2 --labels /content/labels_local.csv --num-workers 12 --resume
```

## Settings

* model: vit_b16, seed: 44
* crop: component, imbalance: class_weights, thicken strokes: False
* max epochs: 25, patience: 5, epoch kept by: val_kappa
* checkpoint evaluated: `vit_b16_v2_s44_best.pt`
* best epoch: 21

## Test results

Test set: 8,938 clocks. 95% intervals from resampling participants.

| | kappa | accuracy | macro F1 |
|---|---|---|---|
| argmax | 0.835 [0.824, 0.845] | 0.716 | 0.665 |
| tuned cut points | 0.852 [0.842, 0.861] | 0.717 | 0.670 |

Cut points picked on validation: 0.01, 1.96, 2.56, 3.38, 4.52

## Paired comparisons

| A | B | kappa A - kappa B (95% CI) |
|---|---|---|
| vit_b16_v2_s44 | vit_b16_v1_s42 | +0.027 [+0.018, +0.035] |

If the interval doesn't cross 0, the difference is real (not noise).

## Every epoch

| epoch | train loss | train acc | val loss | val acc | val kappa |
|---|---|---|---|---|---|
| 0 | 1.0976 | 0.539 | 1.0243 | 0.563 | 0.732 |
| 1 | 0.9222 | 0.624 | 0.8888 | 0.611 | 0.749 |
| 2 | 0.8532 | 0.655 | 0.8409 | 0.643 | 0.781 |
| 3 | 0.8153 | 0.667 | 0.9015 | 0.679 | 0.781 |
| 4 | 0.7803 | 0.678 | 0.8603 | 0.684 | 0.803 |
| 5 | 0.7397 | 0.684 | 0.9859 | 0.705 | 0.815 |
| 6 | 0.7216 | 0.692 | 0.9229 | 0.692 | 0.805 |
| 7 | 0.6821 | 0.700 | 0.9327 | 0.709 | 0.814 |
| 8 | 0.6525 | 0.711 | 1.0909 | 0.701 | 0.819 |
| 9 | 0.6166 | 0.715 | 1.1202 | 0.714 | 0.824 |
| 10 | 0.5881 | 0.722 | 1.1247 | 0.705 | 0.814 |
| 11 | 0.5669 | 0.729 | 1.2022 | 0.716 | 0.822 |
| 12 | 0.5410 | 0.737 | 1.3420 | 0.710 | 0.827 |
| 13 | 0.5123 | 0.741 | 1.4021 | 0.712 | 0.825 |
| 14 | 0.4953 | 0.748 | 1.5163 | 0.715 | 0.825 |
| 15 | 0.4838 | 0.753 | 1.5954 | 0.718 | 0.829 |
| 16 | 0.4549 | 0.758 | 1.5446 | 0.712 | 0.824 |
| 17 | 0.4398 | 0.761 | 1.7206 | 0.707 | 0.815 |
| 18 | 0.4389 | 0.768 | 1.7199 | 0.714 | 0.825 |
| 19 | 0.4180 | 0.773 | 1.6570 | 0.718 | 0.826 |
| 20 | 0.4173 | 0.775 | 1.6538 | 0.717 | 0.829 |
| 21 (kept) | 0.4113 | 0.776 | 1.6948 | 0.721 | 0.830 |
| 22 | 0.4047 | 0.776 | 1.7583 | 0.717 | 0.828 |
| 23 | 0.4018 | 0.779 | 1.7595 | 0.718 | 0.827 |
| 24 | 0.3973 | 0.780 | 1.7576 | 0.718 | 0.829 |

## Files

* `metrics.json`: every number above, as the run wrote it
* `history.json`: every epoch
* `sessions.json` and `train_log.txt`: when it ran and everything it printed
* `predictions_test.csv`, `predictions_val.csv`: one row per clock (has participant IDs, keep it on Drive, never on GitHub)
* `confusion_matrix_normalized.png`, `classification_report.json`


## After this notebook

1. Paste the record above (or the `test kappa` and `[compare]` lines) in the chat.
2. Open `notebooks/06_results_report.ipynb` (CPU is fine) to put all the runs side by side.